# WS/IS MB-OSA and Adaptive Slice-Width Mapping
This notebook analyzes the four canonical WS/IS × optical/digital accumulation structures. The first MRR operand is sliced to 1/2/4 bits, or uses the unsliced 8-bit Analog path. Physical MRR counts and total operand precision remain fixed. Accuracy is **NOT_MODELED**.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
run_dir = Path(os.environ['OPTICALLOOP_MULTISLICE_RUN_DIR']).resolve()
artifacts = run_dir / 'artifacts-multislice'
metadata = json.loads((run_dir / 'run.json').read_text())
fixed = pd.read_csv(artifacts / 'fixed_width_summary.csv')
aswm = pd.read_csv(artifacts / 'aswm_summary.csv')
selections = pd.read_csv(artifacts / 'aswm_layer_selections.csv')
comparison = pd.read_csv(artifacts / 'sensitivity_comparison.csv')
checks = pd.read_csv(artifacts / 'validation.csv')
metadata['tier'], metadata['successful_jobs'], metadata['expected_jobs']

## Reproduced configuration and coverage

In [ ]:
manifest = metadata['manifest']
configuration = pd.DataFrame(manifest['architectures']).set_index('name')
coverage = fixed.groupby(['network', 'variant', 'architecture']).layers.max().unstack('variant')
display(configuration)
display(coverage)
assert metadata['successful_jobs'] == metadata['expected_jobs']
assert checks.loc[checks.status == 'FAIL'].empty

## Validation and accuracy boundary

In [ ]:
display(checks)
assert fixed.accuracy_status.eq('NOT_MODELED').all()
assert aswm.accuracy_status.eq('NOT_MODELED').all()

## Primary 5.2 pJ/bit model

In [ ]:
primary_fixed = fixed[(fixed.energy_model == 'linear_bit') & (fixed.optical_loss_db_per_stage == 0)]
primary_aswm = aswm[(aswm.energy_model == 'linear_bit') & (aswm.optical_loss_db_per_stage == 0)]
headline = comparison[(comparison.energy_model == 'linear_bit') & (comparison.optical_loss_db_per_stage == 0)]
display(headline.sort_values(['mapping', 'network', 'edp_j_s']))
display(primary_fixed.sort_values(['stationarity', 'network', 'front_mrr_slice_bits']))

## Energy-delay and slice selections

In [ ]:
ax = primary_fixed.plot.scatter(x='latency_s', y='energy_j', c='front_mrr_slice_bits', colormap='viridis', logx=True, logy=True, figsize=(8, 6))
ax.set_title('WS/IS fixed-width energy-delay points')
plt.show()
primary_selections = selections[(selections.energy_model == 'linear_bit') & (selections.optical_loss_db_per_stage == 0)]
selection_counts = primary_selections.groupby(['mapping', 'stationarity', 'front_mrr_slice_bits']).size().unstack(fill_value=0)
display(selection_counts)
selection_counts.plot.bar(stacked=True, figsize=(11, 5), title='Adaptive front-MRR slice and stationarity selections')
plt.show()

## Sensitivity boundary
The optimistic constant-symbol and conservative Walden DAC models, together with 0/0.5/1 dB optical-loss compensation per delay stage, are modeling sensitivities—not measured accuracy results.

In [ ]:
sensitivity = aswm.groupby(['energy_model', 'optical_loss_db_per_stage']).edp_j_s.agg(['min', 'median', 'max'])
display(sensitivity)